In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
import torchmetrics as tm
import torch.nn.functional as F
import seaborn as sns
import scipy.stats as stats
import plotly.express as px
from pathlib import Path
import umap
from sklearn.decomposition import PCA
from statsmodels.stats.multitest import multipletests
from scipy.stats import wilcoxon
import plotly.graph_objects as go
from utilities import precision_at_q, ndcg_at_q
torch.manual_seed(9497)

# Model 

In [2]:
from exp_regression import DrugAware_DWM, Transcriptomic_Head, SMILES_Head, CombinedModel
from cli_regressor import Module_training_reg
from utilities import CVDataModule

In [3]:
tx_head = Transcriptomic_Head(
            input_size=946, # 946 | 4377
            hd_1=512,
            hd_2=256
        )

chem_head = SMILES_Head(
    input_size=512,
    hd_1=512, 
    hd_2=256,
    hd_3=128
)

combined_model = CombinedModel(
    input_size=1024,
    hd_1=256, #256
    hd_2=128  #128
)

loss_fn = nn.MSELoss()
lr= 1e-4
wd=0.0

# Data preparation

In [4]:
gdsc_gex = pd.read_csv(
    "output/regression/L1000_GEX_data_filtered_logCPM.csv",
    index_col=0
)

In [5]:
gdsc_gex.head()

,ABCB6,ABCC5,ABCF1,ABCF3,ABHD4,ABHD6,ABL1,ACAA1,ACAT2,ACBD3,...,ZMIZ1,ZMYM2,ZNF131,ZNF274,ZNF318,ZNF395,ZNF451,ZNF586,ZNF589,ZW10
model_name,,,,,,,,,,,,,,,,,,,,,
22RV1,6.692272,5.974217,5.972175,5.937373,3.905459,2.274118,6.491253,5.258903,6.844647,6.636650,...,4.987584,6.736658,5.452852,4.495404,6.299981,5.998490,6.188324,4.225246,4.699200,5.302827
23132-87,5.811651,5.621582,6.928492,5.869526,6.295594,3.886941,5.921526,5.789106,5.768256,5.802125,...,5.706728,6.679969,5.640479,4.667580,5.517527,5.866943,6.234096,3.788737,4.580018,5.407221
42-MG-BA,6.339612,4.853004,7.245003,5.280296,5.325799,4.142516,7.026414,5.161804,6.125542,6.124434,...,7.336627,4.794387,5.917842,5.280881,4.977325,5.281349,5.257054,2.324235,2.769024,5.603927
451Lu,5.219828,5.896027,6.377211,5.705629,4.042942,4.193596,5.952298,5.341295,5.092637,6.007741,...,6.561371,5.502658,4.866528,3.187331,5.518168,5.420912,4.847967,3.257737,4.093944,4.471016
5637,3.049106,4.826572,7.045027,5.333473,3.510659,1.870316,7.449089,5.054642,6.051388,5.287819,...,6.087544,5.290822,5.164975,4.105767,4.963710,6.145299,4.879865,3.134558,2.752614,5.516239


In [6]:
pdo_gex = pd.read_csv(
    "data/validation_data/PDOs/integrated_data/pdo_logCPM.csv",
    index_col=0
)

In [7]:
pdo_gex.head()

,ISG15,UBE2J2,C1orf159,TTLL10,ACAP3,CCNL2,ATAD3C,ATAD3B,ATAD3A,SSU72,...,NDUFB2-AS1,RLN2,SUGT1P1,TKTL1,MGAT3,RFX6,CALML5,DNAJB13,NPY1R,TAGAP
P9-M-LN,2.135407,5.956173,3.865369,0.000000,5.421376,6.602177,3.482956,4.871848,4.755832,6.345458,...,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
P20-T-G,5.641111,6.297789,3.801721,3.267579,5.367185,6.822650,4.496700,3.943341,4.824220,6.876626,...,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
P47-T-T,4.193460,7.046161,4.302859,1.287672,5.093846,6.931593,1.957051,5.513537,6.072567,7.356418,...,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
P47-R-AP,4.294714,6.575511,3.991047,1.164626,4.466613,6.575511,3.991047,5.001973,6.190251,7.365546,...,1.164626,1.164626,1.164626,1.164626,0.0,0.0,0.0,0.0,0.0,0.0
P48-M-LN,1.675951,3.582133,4.214310,6.718042,4.714022,6.940625,6.687640,5.260028,5.218100,7.418996,...,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0


In [8]:
len(pdo_gex.columns.intersection(gdsc_gex.columns))

901

In [9]:
pdo_gex = pdo_gex.loc[
    :,pdo_gex.columns.intersection(gdsc_gex.columns)]

# reordering genes by training order
pdo_gex = pdo_gex.reindex(columns=gdsc_gex.columns, fill_value=0.0)
pdo_gex.shape

(54, 946)

In [10]:
pdo_dr = pd.read_csv(
    "data/validation_data/PDOs/integrated_data/dose_response_pdo.csv"
)

In [11]:
pdo_dr["Y_TRUE"] = (
    pdo_dr.groupby("Line")["LogIC50"]
    .transform(lambda x: stats.zscore(x, nan_policy="omit"))
)

In [12]:
pdo_dr = pdo_dr.loc[pdo_dr.seen_before=="yes",:]

In [13]:
pdo_dr.head()

,Line,Drug,LogIC50,PATHWAY_NAME,seen_before,Lab,TCGA_DESC,Y_TRUE
0,P9-M-LN,Cisplatin,2.165619,DNA replication,yes,KK,HNSC,-0.404912
1,P20-T-G,Cisplatin,1.249902,DNA replication,yes,KK,HNSC,-1.353180
2,P47-T-T,Cisplatin,1.047319,DNA replication,yes,KK,HNSC,-1.000000
3,P47-R-AP,Cisplatin,1.420696,DNA replication,yes,KK,HNSC,-0.110678
4,P48-M-LN,Cisplatin,1.667707,DNA replication,yes,KK,HNSC,-1.602422


In [14]:
smiles = pd.read_csv(
    "data/validation_data/PDOs/integrated_data/vector_smiles.csv",
    index_col=0
)

In [15]:
smiles.head()

,0,1,2,3,4,5,6,7,8,9,...,502,503,504,505,506,507,508,509,510,511
Entinostat,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
SB216763,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
Taselisib,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0
Axitinib,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Camptothecin,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


# Cross validation | Fixed-drug and Fixed-line

In [16]:
def covariance_sqrt(X, eps=1e-3, shrink=0.1):
    C = np.cov(
        X,
        rowvar=False
    )
    tr_over_d = np.trace(C) / C.shape[0]
    C = (1-shrink)*C + shrink*tr_over_d*np.eye(C.shape[0])
    
    C = C + eps*np.eye(C.shape[0])

    U, S, Vt = np.linalg.svd(
        C, full_matrices=False
    )
    A = (U*np.sqrt(S)) @ Vt

    return A

def CORAL_transform(S, T):
    #-----source-----
    mu_s = S.mean(0)
    A_s = covariance_sqrt(S)

    #----target----
    mu_t = T.mean(0)
    A_t = covariance_sqrt(T)
    A_t_inv = np.linalg.inv(A_t)

    A = A_t_inv @ A_s
    T_coral = (T - mu_t) @ A + mu_s

    return T_coral


In [50]:

root = "output/regression/cross_validation"
cancer_type = "pancancer" # pancancer , solid_tumors
experiment = "NBS_cells"

fold_dirs = Path(f"{root}/{cancer_type}/{experiment}/")
ckpt_dir = Path(f"cDWM/L1000_{cancer_type}_NBS_cells/")
# ckpt_dir = Path(f"naive_predictor/L1000_{cancer_type}_NBS_cells/")

# Sort fold paths in numeric order (fold_0, fold_1, ...)
folds = sorted(fold_dirs.glob("fold_*"), key=lambda x: int(x.name.split("_")[1]))
ckpts = sorted(ckpt_dir.glob("version_*"), key=lambda x: int(x.name.split("_")[1]))

per_drug_pcc_cv = {}
per_line_pcc_cv = {}

per_drug_precision_cv = {}
per_cell_precision_cv = {}

per_drug_ndcg_cv = {}
per_cell_ndcg_cv = {}


for fold_path, ckpt_path in zip(folds, ckpts):
    fold_name = fold_path.name.split("_")[1]
    ckpt_file = list((ckpt_path / 'checkpoints').glob('*.ckpt'))[0]

    # Prediction
    regressor = Module_training_reg.load_from_checkpoint(
    ckpt_file,
    map_location="cpu",
    tx_nn=tx_head,
    chem_nn=chem_head,
    comb_nn=combined_model,
    loss_fn=loss_fn,
    lr=lr,
    weight_decay=wd
    )

    data_module = CVDataModule(
        root = root,
        type = cancer_type,
        experiment = experiment,
        fold_n = fold_name,
        GEX_path = "output/regression/L1000_GEX_data_filtered_logCPM.csv",
        SMILES_path = "output/regression/vector_smiles_512.csv",
        batch_size = 512,
        num_workers = 12
    )

    data_module.setup()
    

    ########################################
    #
    # SUBSETTING TO PREDICTING SETS
    #
    ########################################
    
    train_mean = torch.tensor(data_module.gex_mean.to_numpy(), dtype=torch.float32)
    train_std = torch.tensor(data_module.gex_std.to_numpy(), dtype=torch.float32)
    
    gex = torch.tensor(
        pdo_gex.loc[pdo_dr.Line,:].to_numpy(), dtype=torch.float32
    )
    
    sm = torch.tensor(
        smiles.loc[pdo_dr.Drug, :].to_numpy(), dtype=torch.float32
    )

    
    #-----standard prediction-----
    gex = (gex - train_mean)/train_std
    y_hat = regressor.predict(
        gex=gex,
        sm=sm
    )

    
    y_hat = y_hat.double()
    y_hat = y_hat.numpy()
    pdo_dr["Y_HAT"] = y_hat

    # Computing metrics
    per_drug_pcc = (pdo_dr.groupby("Drug")
                    .filter(lambda x: len(x) >=2)
                    .groupby("Drug")
                    .apply(lambda x: pd.Series(stats.pearsonr(x["Y_TRUE"], x["Y_HAT"]), index=["PCC", "p_value"]))
                    .reset_index())

    per_drug_pcc_cv[fold_name] = [per_drug_pcc["PCC"].median()]

    per_line_pcc = (pdo_dr.groupby("Line")
                    .filter(lambda x: len(x) >=2)
                    .groupby("Line")
                    .apply(lambda x: pd.Series(stats.pearsonr(x["Y_TRUE"], x["Y_HAT"]), index=["PCC", "p_value"]))
                    .reset_index())

    per_line_pcc_cv[fold_name] = [per_line_pcc["PCC"].median()]


    ##############################
    # PRECISION
    ##############################
    per_drug_pcc = (pdo_dr.groupby("Drug")
                    .filter(lambda x: len(x) >=2)
                    .groupby("Drug")
                    .apply(lambda x: pd.Series(precision_at_q(x["Y_TRUE"], x["Y_HAT"]), index=["precision@q25"]))
                    # .reset_index()
                   )

    per_drug_precision_cv[fold_name] = [per_drug_pcc["precision@q25"].median()]

    per_cell_pcc = (pdo_dr.groupby("Line")
                    .filter(lambda x: len(x) >=2)
                    .groupby("Line")
                    .apply(lambda x: pd.Series(precision_at_q(x["Y_TRUE"], x["Y_HAT"]), index=["precision@q25"]))
                    # .reset_index()
                   )

    per_cell_precision_cv[fold_name] = [per_cell_pcc["precision@q25"].median()]

    ##############################
    # NDCG
    ##############################
    per_drug_pcc = (pdo_dr.groupby("Drug")
                    .filter(lambda x: len(x) >=2)
                    .groupby("Drug")
                    .apply(lambda x: pd.Series(ndcg_at_q(x["Y_TRUE"], x["Y_HAT"]), index=["ndcg@q25"]))
                    # .reset_index()
                   )

    per_drug_ndcg_cv[fold_name] = [per_drug_pcc["ndcg@q25"].median()]

    per_cell_pcc = (pdo_dr.groupby("Line")
                    .filter(lambda x: len(x) >=2)
                    .groupby("Line")
                    .apply(lambda x: pd.Series(ndcg_at_q(x["Y_TRUE"], x["Y_HAT"]), index=["ndcg@q25"]))
                    # .reset_index()
                   )

    per_cell_ndcg_cv[fold_name] = [per_cell_pcc["ndcg@q25"].median()]



/tmp/ipykernel_9307/4229887775.py:87: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series(stats.pearsonr(x["Y_TRUE"], x["Y_HAT"]), index=["PCC", "p_value"]))
/tmp/ipykernel_9307/4229887775.py:95: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series(stats.pearsonr(x["Y_TRUE"], x["Y_HAT"]), index=["PCC", "p_value"]))
/tmp/ipykernel_9307/4229887775.py:107: FutureWarning: DataFram

In [51]:
per_drug_pcc_cv = pd.DataFrame(per_drug_pcc_cv)
per_drug_pcc_cv.index = ["DWM-NELLY"]
per_drug_pcc_cv

,0,1,2,3,4,5,6,7,8,9
DWM-NELLY,0.080257,0.131994,0.069645,0.108124,0.080136,0.084387,0.051565,0.052461,0.078169,0.051811


In [52]:
M = np.median(per_drug_pcc_cv)
sem = stats.sem(per_drug_pcc_cv.to_numpy().flatten())
ci = stats.t.interval(0.95, df=9, loc=M, scale=sem)
print(f"Median={M}, low={ci[0]}, high={ci[1]}")

Median=0.07915251984961974, low=0.06072426277758313, high=0.09758077692165634


In [53]:
per_line_pcc_cv = pd.DataFrame(per_line_pcc_cv)
per_line_pcc_cv.index = ["DWM-NELLY"]
per_line_pcc_cv

,0,1,2,3,4,5,6,7,8,9
DWM-NELLY,0.721992,0.719065,0.703666,0.711433,0.745638,0.709545,0.740045,0.739978,0.746362,0.739592


In [54]:
M = np.median(per_line_pcc_cv)
sem = stats.sem(per_line_pcc_cv.to_numpy().flatten())
ci = stats.t.interval(0.95, df=9, loc=M, scale=sem)
print(f"Median={M}, low={ci[0]}, high={ci[1]}")

Median=0.7307918990254404, low=0.7191258235471287, high=0.7424579745037521


In [ ]:
# per_drug_pcc_cv.to_csv(
#     f"cDWM/L1000_{cancer_type}_{experiment}/pdo_fixed-drug_CV.csv"
# )

In [24]:
# per_line_pcc_cv.to_csv(
#     f"cDWM/L1000_{cancer_type}_{experiment}/pdo_fixed-line_CV.csv"
# )

In [ ]:
# per_drug_pcc_cv.to_csv(
#     f"naive_predictor/L1000_{cancer_type}_{experiment}/pdo_fixed-drug_CV.csv"
# )

In [ ]:
# per_line_pcc_cv.to_csv(
#     f"naive_predictor/L1000_{cancer_type}_{experiment}/pdo_fixed-line_CV.csv"
# )

In [55]:
per_drug_precision_cv = pd.DataFrame(per_drug_precision_cv)
per_drug_precision_cv.index = ["DWM-NELLY"]
per_drug_precision_cv

,0,1,2,3,4,5,6,7,8,9
DWM-NELLY,0.333333,0.333333,0.285714,0.333333,0.333333,0.333333,0.333333,0.333333,0.333333,0.333333


In [56]:
M = np.median(per_drug_precision_cv)
sem = stats.sem(per_drug_precision_cv.to_numpy().flatten())
ci = stats.t.interval(0.95, df=9, loc=M, scale=sem)
print(f"Median={M}, low={ci[0]}, high={ci[1]}")

Median=0.3333333333333333, low=0.32256115636736143, high=0.3441055102993052


In [ ]:
# per_drug_precision_cv.to_csv(
#     "naive_predictor/L1000_pancancer_NBS_cells/precision_pdo_fixed-drug_CV.csv"
# )

In [ ]:
# per_drug_precision_cv.to_csv(
#     "cDWM/L1000_pancancer_NBS_cells/precision_pdo_fixed-drug_CV.csv"
# )

In [57]:
per_drug_ndcg_cv = pd.DataFrame(per_drug_ndcg_cv)
per_drug_ndcg_cv.index = ["DWM-NELLY"]
per_drug_ndcg_cv

,0,1,2,3,4,5,6,7,8,9
DWM-NELLY,0.0,0.0,0.0,0.003189,0.044223,0.0,0.0,0.0,0.10177,0.0


In [58]:
M = np.median(per_drug_ndcg_cv)
sem = stats.sem(per_drug_ndcg_cv.to_numpy().flatten())
ci = stats.t.interval(0.95, df=9, loc=M, scale=sem)
print(f"Median={M}, low={ci[0]}, high={ci[1]}")

Median=0.0, low=-0.02396114294802275, high=0.02396114294802275


In [ ]:
# per_drug_ndcg_cv.to_csv(
#     "naive_predictor/L1000_pancancer_NBS_cells/ndgc_pdo_fixed-drug_CV.csv"
# )

In [ ]:
# per_drug_ndcg_cv.to_csv(
#     "cDWM/L1000_pancancer_NBS_cells/ndgc_pdo_fixed-drug_CV.csv"
# )

In [59]:
per_cell_precision_cv = pd.DataFrame(per_cell_precision_cv)
per_cell_precision_cv.index = ["DWM-NELLY"]
per_cell_precision_cv

,0,1,2,3,4,5,6,7,8,9
DWM-NELLY,0.611111,0.611111,0.611111,0.611111,0.666667,0.611111,0.611111,0.611111,0.666667,0.611111


In [60]:
M = np.median(per_cell_precision_cv)
sem = stats.sem(per_cell_precision_cv.to_numpy().flatten())
ci = stats.t.interval(0.95, df=9, loc=M, scale=sem)
print(f"Median={M}, low={ci[0]}, high={ci[1]}")

Median=0.6111111111111112, low=0.594354391386266, high=0.6278678308359563


In [ ]:
# per_cell_precision_cv.to_csv(
#     f"naive_predictor/L1000_{cancer_type}_{experiment}/precision_pdo_fixed-line_CV.csv"
# )

In [ ]:
# per_cell_precision_cv.to_csv(
#     f"cDWM/L1000_{cancer_type}_{experiment}/precision_pdo_fixed-line_CV.csv"
# )

In [61]:
per_cell_ndcg_cv = pd.DataFrame(per_cell_ndcg_cv)
per_cell_ndcg_cv.index = ["DWM-NELLY"]
per_cell_ndcg_cv

,0,1,2,3,4,5,6,7,8,9
DWM-NELLY,0.751394,0.724845,0.785638,0.729603,0.797533,0.772928,0.761222,0.765154,0.797533,0.761222


In [62]:
M = np.median(per_cell_ndcg_cv)
sem = stats.sem(per_cell_ndcg_cv.to_numpy().flatten())
ci = stats.t.interval(0.95, df=9, loc=M, scale=sem)
print(f"Median={M}, low={ci[0]}, high={ci[1]}")

Median=0.7631883357913988, low=0.7452111338409076, high=0.7811655377418899


In [ ]:
# per_cell_ndcg_cv.to_csv(
#     f"naive_predictor/L1000_{cancer_type}_{experiment}/ndcg_pdo_fixed-line_CV.csv"
# )

In [ ]:
# per_cell_ndcg_cv.to_csv(
#     f"cDWM//L1000_{cancer_type}_{experiment}/ndcg_pdo_fixed-line_CV.csv"
# )

In [32]:
pdo_dr.Y_TRUE[pdo_dr.TCGA_DESC=="PDAC"].corr(pdo_dr.Y_HAT[pdo_dr.TCGA_DESC=="PDAC"], method="pearson")

np.float64(0.8371003694192835)

In [33]:
pdo_dr.TCGA_DESC.unique()

array(['HNSC', 'BLCA', 'PDAC', 'COAD'], dtype=object)

In [23]:
fig = px.scatter(
    x=pdo_dr.Y_TRUE,
    y=pdo_dr.Y_HAT,
    facet_col=pdo_dr.TCGA_DESC,
    color=pdo_dr.TCGA_DESC,
    # opacity=0.7,
    symbol=pdo_dr.TCGA_DESC,
    height=400,
    width=1000,
    template="simple_white",
    # trendline="ols",
    labels={"y": "Predicted Z-score IC50",
            "x": "Observed Z-score IC50",
            "color" : "Cancer type"
           },
)


fig.update_traces(
    marker=dict(
        size=8,
        line=dict(width=1.5, color="DarkSlateGrey")
    )
)

fig.update_traces(
    line=dict(color="black", width=2),
    selector=dict(mode="lines")
)

pcc = pdo_dr.Y_TRUE[pdo_dr.TCGA_DESC=="HNSC"].corr(pdo_dr.Y_HAT[pdo_dr.TCGA_DESC=="HNSC"], method="pearson")
fig.add_annotation(
    x=0.03, y=1.1,                  # position inside panel (3% from left, 95% from bottom)
    xref=f"x domain",
    yref=f"y domain",
    text=f"PCC = {pcc:.2f}",
    showarrow=False,
    align="left",
    font=dict(size=12)
)


pcc = pdo_dr.Y_TRUE[pdo_dr.TCGA_DESC=="BLCA"].corr(pdo_dr.Y_HAT[pdo_dr.TCGA_DESC=="BLCA"], method="pearson")
fig.add_annotation(
    x=0.03, y=1.1,                  # position inside panel (3% from left, 95% from bottom)
    xref=f"x2 domain",
    yref=f"y2 domain",
    text=f"PCC = {pcc:.2f}",
    showarrow=False,
    align="left",
    font=dict(size=12)
)

pcc = pdo_dr.Y_TRUE[pdo_dr.TCGA_DESC=="PDAC"].corr(pdo_dr.Y_HAT[pdo_dr.TCGA_DESC=="PDAC"], method="pearson")
fig.add_annotation(
    x=0.03, y=1.1,                  # position inside panel (3% from left, 95% from bottom)
    xref=f"x3 domain",
    yref=f"y3 domain",
    text=f"PCC = {pcc:.2f}",
    showarrow=False,
    align="left",
    font=dict(size=12)
)

pcc = pdo_dr.Y_TRUE[pdo_dr.TCGA_DESC=="COAD"].corr(pdo_dr.Y_HAT[pdo_dr.TCGA_DESC=="COAD"], method="pearson")
fig.add_annotation(
    x=0.03, y=1.1,                  # position inside panel (3% from left, 95% from bottom)
    xref=f"x4 domain",
    yref=f"y4 domain",
    text=f"PCC = {pcc:.2f}",
    showarrow=False,
    align="left",
    font=dict(size=12)
)


fig.for_each_annotation(
    lambda a: a.update(text="") if isinstance(a.text, str) and a.text.startswith("facet_col=") else None
)

fig

In [58]:
# fig.write_image(
#     "slides/figures_regression/benchmark/ic50/pred_vs_observed.pdf"
# )

In [19]:
tmp = pdo_dr.loc[pdo_dr.TCGA_DESC=="BLCA"]

# tmp = pdo_dr.loc[(pdo_dr.TCGA_DESC=="HNSC")&(pdo_dr.Lab=="HC")]

fig = px.scatter(
    data_frame=tmp,
    x="Y_TRUE",
    y="Y_HAT",
    facet_col="Line",
    color="Line",
    # opacity=0.7,
    # symbol=pdo_dr.TCGA_DESC,
    height=400,
    width=1000,
    template="simple_white",
    # trendline="ols",
    labels={"Y_HAT": "Predicted IC50",
            "Y_TRUE": "Observed IC50",
            "color" : "Organoid line"
           },
)


fig.update_traces(
    marker=dict(
        size=8,
        line=dict(width=1.5, color="DarkSlateGrey")
    )
)

fig.update_traces(
    line=dict(color="black", width=2),
    selector=dict(mode="lines")
)

# add PCC + SCC per facet
for i, (ct, df_sub) in enumerate(tmp.groupby("Line"), start=1):
    pcc = df_sub["Y_TRUE"].corr(df_sub["Y_HAT"], method="pearson")
    scc = df_sub["Y_TRUE"].corr(df_sub["Y_HAT"], method="spearman")

    # axis ids: first facet uses 'x','y', others 'x2','y2',...
    ax_id = "" if i == 1 else str(i)

    fig.add_annotation(
        x=0.03, y=1.1,                  # position inside panel (3% from left, 95% from bottom)
        xref=f"x{ax_id} domain",
        yref=f"y{ax_id} domain",
        text=f"PCC = {pcc:.2f}<br>SCC = {scc:.2f}",
        showarrow=False,
        align="left",
        font=dict(size=12)
    )

fig.for_each_annotation(
    lambda a: a.update(text="") if isinstance(a.text, str) and a.text.startswith("Line=") else None
)

fig

In [48]:
# fig.write_image(
#     "slides/figures_regression/benchmark/ic50/pred_vs_observed_BLCA.pdf"
# )

# Cross validation | Fixed-cancer type and Fixed-lab

In [ ]:

root = "output/regression/cross_validation"
cancer_type = "pancancer" # pancancer , solid_tumors
experiment = "NBS_cells"

fold_dirs = Path(f"{root}/{cancer_type}/{experiment}/")
ckpt_dir = Path(f"cDWM/L1000_{cancer_type}_NBS_cells/")
# ckpt_dir = Path(f"naive_predictor/L1000_{cancer_type}_NBS_cells/")

# Sort fold paths in numeric order (fold_0, fold_1, ...)
folds = sorted(fold_dirs.glob("fold_*"), key=lambda x: int(x.name.split("_")[1]))
ckpts = sorted(ckpt_dir.glob("version_*"), key=lambda x: int(x.name.split("_")[1]))

per_cancer_pcc_cv = {}
per_lab_pcc_cv = {}


for fold_path, ckpt_path in zip(folds, ckpts):
    fold_name = fold_path.name.split("_")[1]
    ckpt_file = list((ckpt_path / 'checkpoints').glob('*.ckpt'))[0]

    regressor = Module_training_reg.load_from_checkpoint(
    ckpt_file,
    map_location="cpu",
    tx_nn=tx_head,
    chem_nn=chem_head,
    comb_nn=combined_model,
    loss_fn=loss_fn,
    lr=lr,
    weight_decay=wd
    )

    data_module = CVDataModule(
        root = root,
        type = cancer_type,
        experiment = experiment,
        fold_n = fold_name,
        GEX_path = "output/regression/L1000_GEX_data_filtered_logCPM.csv",
        SMILES_path = "output/regression/vector_smiles_512.csv",
        batch_size = 512,
        num_workers = 12
    )


    data_module.setup()

    blood = ["LCML", "CLL", "MM", "LAML", "ALL", "DLBC", "UNCLASSIFIED"]
    train_set = data_module._load_set(set_name = "train")
    train_set = (
        train_set.loc[~train_set.TCGA_DESC.isin(blood),:]
        .drop_duplicates(subset="CELL_LINE_NAME")
    )

    ########################################
    #
    # SUBSETTING TO PREDICTING SETS
    #
    ########################################
    train_mean = torch.tensor(data_module.gex_mean.to_numpy(), dtype=torch.float32)
    train_std = torch.tensor(data_module.gex_std.to_numpy(), dtype=torch.float32)
    
    gex = torch.tensor(
        pdo_gex.loc[pdo_dr.Line,:].to_numpy(), dtype=torch.float32
    )
    
    sm = torch.tensor(
        smiles.loc[pdo_dr.Drug, :].to_numpy(), dtype=torch.float32
    )

    gex = (gex - train_mean)/train_std
    #----STANDARD PREDICTION----
    y_hat = regressor.predict(
        gex=gex,
        sm=sm
    )
    
    y_hat = y_hat.double()
    y_hat = y_hat.numpy()
    pdo_dr["Y_HAT"] = y_hat

    cancer_type = (
        pdo_dr[["Line","TCGA_DESC"]]
        .drop_duplicates()
    )

    labs = (
        pdo_dr[["Line", "Lab"]]
        .drop_duplicates()
    )


    # Computing metrics
    per_cancer_pcc = (pdo_dr.groupby("TCGA_DESC")
                    .filter(lambda x: len(x) >=2)
                    .groupby("TCGA_DESC")
                    .apply(lambda x: pd.Series(stats.pearsonr(x["Y_TRUE"], x["Y_HAT"]), index=["PCC", "p_value"]))
                    .reset_index())
    per_cancer_pcc = (per_cancer_pcc
                      .merge(cancer_type, on="TCGA_DESC", how="left")
                      .dropna(subset=["TCGA_DESC"])
                      .groupby("TCGA_DESC")["PCC"]
                      .median()
                     )

    per_cancer_pcc_cv[fold_name] = per_cancer_pcc

    per_lab_pcc = (pdo_dr.groupby("Lab")
                    .filter(lambda x: len(x) >=2)
                    .groupby("Lab")
                    .apply(lambda x: pd.Series(stats.pearsonr(x["Y_TRUE"], x["Y_HAT"]), index=["PCC", "p_value"]))
                    .reset_index())
    per_lab_pcc = (per_lab_pcc
                   .merge(labs, on="Lab", how="left")
                   .dropna(subset=["Lab"])
                   .groupby("Lab")["PCC"]
                   .median()
                   )

    per_lab_pcc_cv[fold_name] = per_lab_pcc



In [28]:
cancer_type = (pd.DataFrame(per_cancer_pcc_cv)
               .groupby(level=0)
               .median()
               .median(axis=1)
               .rename_axis("")
               .T
)
cancer_type.name = "naive-NELLY"

In [29]:
cancer_type


BLCA    0.426146
COAD    0.621082
HNSC    0.400610
PDAC    0.723024
Name: naive-NELLY, dtype: float64

In [ ]:
# cancer_type.to_csv(
#     "cDWM/L1000_pancancer_NBS_cells/pdo_fixed-cancer_CV.csv"
# )

In [ ]:
# cancer_type.to_csv(
#     "naive_predictor/L1000_pancancer_NBS_cells/pdo_fixed-cancer_CV.csv"
# )

In [42]:
lab = (pd.DataFrame(per_lab_pcc_cv)
               .groupby(level=0)
               .median()
               .median(axis=1)
               .rename_axis("")
               .T
)

In [43]:
lab.name = "DWM-NELLY"

In [44]:
lab


HC    0.750046
KK    0.401712
MS    0.268545
Name: DWM-NELLY, dtype: float64

In [84]:
lab


HC    0.662068
KK    0.402901
MS    0.446153
Name: DWM-NELLY, dtype: float64

In [ ]:
# lab.to_csv(
#     "cDWM/L1000_pancancer_NBS_cells/pdo_fixed-lab_CV.csv"
# )

# Model comparison

## NBS cell | Fixed-drug

### PCC

In [2]:
screendl_fixed_drug = pd.read_csv(
    "benchmarking/screendl/predictions_pancancer_NBS_cells/pdo_fixed-drug_evaluation.csv",
    header=0,
    index_col=0
)

In [3]:
screendl_fixed_drug.head()

,fold_0,fold_1,fold_2,fold_3,fold_4,fold_5,fold_6,fold_7,fold_8,fold_9
ScreenDL,-0.15609,0.024,-0.015163,-0.165738,0.309622,0.190785,0.296507,-0.031808,-0.045203,0.139611


In [4]:
hidra_fixed_drug = pd.read_csv(
    "benchmarking/hidra/CV/predictions_pancancer_NBS_cells/pdo_fixed-drug_CV.csv",
    index_col=0
)

In [5]:
hidra_fixed_drug.head()

,fold_0,fold_1,fold_2,fold_3,fold_4,fold_5,fold_6,fold_7,fold_8,fold_9
HiDRA,0.04103,0.017061,-0.159471,-0.232416,-0.130276,-0.177953,-0.002549,0.130049,0.128656,0.053085


In [6]:
paccmann_fixed_drug = pd.read_csv(
    "benchmarking/paccmann_predictor/CV/predictions_pancancer_NBS_cells/pdo_fixed-drug_CV.csv",
    index_col=0
)

In [7]:
paccmann_fixed_drug.head()

,fold_0,fold_1,fold_2,fold_3,fold_4,fold_5,fold_6,fold_7,fold_8,fold_9
Paccmann,0.010746,0.034388,-0.017473,0.098403,0.109646,0.056463,0.04847,-0.009308,0.217244,-0.023822


In [8]:
naive_NELLY_fixed_drug = pd.read_csv(
    "naive_predictor/L1000_pancancer_NBS_cells/pdo_fixed-drug_CV.csv",
    index_col=0
)

In [9]:
naive_NELLY_fixed_drug.columns = screendl_fixed_drug.columns

In [10]:
naive_NELLY_fixed_drug.head()

,fold_0,fold_1,fold_2,fold_3,fold_4,fold_5,fold_6,fold_7,fold_8,fold_9
naive-NELLY,0.003762,-0.010086,-0.101465,0.187702,-0.035831,0.07708,-0.157887,-0.095954,-0.015763,-0.160902


In [11]:
dwm_NELLY_fixed_drug = pd.read_csv(
    "cDWM/L1000_pancancer_NBS_cells/pdo_fixed-drug_CV.csv",
    index_col=0
)

In [12]:
dwm_NELLY_fixed_drug.columns = screendl_fixed_drug.columns

In [13]:
dwm_NELLY_fixed_drug.head()

,fold_0,fold_1,fold_2,fold_3,fold_4,fold_5,fold_6,fold_7,fold_8,fold_9
DWM-NELLY,0.145308,-0.031882,0.105539,-0.00561,-0.099662,0.028141,0.18618,0.185415,-0.111725,0.189545


In [17]:
metrics_fixed_drug = pd.concat(
    [
        naive_NELLY_fixed_drug,
        screendl_fixed_drug,
        hidra_fixed_drug,
        paccmann_fixed_drug,
        dwm_NELLY_fixed_drug
    ]
).T

In [18]:
metrics_fixed_drug.reset_index(inplace=True)

In [19]:
metrics_fixed_drug

,index,naive-NELLY,ScreenDL,HiDRA,Paccmann,DWM-NELLY
0,fold_0,0.003762,-0.156090,0.041030,0.010746,0.145308
1,fold_1,-0.010086,0.024000,0.017061,0.034388,-0.031882
2,fold_2,-0.101465,-0.015163,-0.159471,-0.017473,0.105539
3,fold_3,0.187702,-0.165738,-0.232416,0.098403,-0.005610
4,fold_4,-0.035831,0.309622,-0.130276,0.109646,-0.099662
5,fold_5,0.077080,0.190785,-0.177953,0.056463,0.028141
6,fold_6,-0.157887,0.296507,-0.002549,0.048470,0.186180
7,fold_7,-0.095954,-0.031808,0.130049,-0.009308,0.185415
8,fold_8,-0.015763,-0.045203,0.128656,0.217244,-0.111725
9,fold_9,-0.160902,0.139611,0.053085,-0.023822,0.189545


In [20]:
metrics_fixed_drug = metrics_fixed_drug.melt(
    id_vars="index",
    var_name="Method",
    value_name="MCorrelation"
)

In [21]:
metrics_fixed_drug.head()

,index,Method,MCorrelation
0,fold_0,naive-NELLY,0.003762
1,fold_1,naive-NELLY,-0.010086
2,fold_2,naive-NELLY,-0.101465
3,fold_3,naive-NELLY,0.187702
4,fold_4,naive-NELLY,-0.035831


In [22]:
palette = {
    "Paccmann": "#1F77B4",       # blue
    "HiDRA": "#FF7F0E",  # orange
    "ScreenDL": "#2CA02C",     # green
    "naive-NELLY": "#D62728",   # red
    "DWM-NELLY" : "#b38ce3"
}

# _, p = wilcoxon(per_drug_pcc_cv.T["CPV"], screendl_fixed_drug.T["ScreenDL"], alternative="greater")
fig = px.violin(metrics_fixed_drug,
                x="Method",
                y="MCorrelation",
                color="Method",
                # height=700,
                # width=700,
                template="simple_white",
                box=False,
                points="all",
                labels={"Method": "",
                        "MCorrelation": "Pearson Correlation"
                       },
                color_discrete_map=palette,
                title= "CV on organoid dataset | Fixed-drug Correlation"
                
                
         )


# Step 2: Add paired-line traces manually
# df_wide = metrics_fixed_drug.pivot(index="index", columns="Method", values="MCorrelation").reset_index()
# for i, row in df_wide.iterrows():
#     fig.add_trace(go.Scatter(
#         x=["CPV", "ScreenDL"],
#         y=[row["CPV"], row["ScreenDL"]],
#         mode="lines+markers",
#         line=dict(color="gray", width=1),
#         marker=dict(size=6, color="black"),
#         name=row["index"],
#         showlegend=False  # hide fold labels
#     ))

fig.update_traces(jitter=0.05, pointpos=0)

# fig.add_annotation(
#     x=0.5,  # Adjust if you have multiple violins (e.g., grouped by drug)
#     y=1,
#     text=f"Wilcoxon signed-rank test, p-value = {p:.2e}",
#     xref="paper", yref="paper",
#     showarrow=False,
#     yanchor='bottom',
#     font=dict(color="black", size=14)
# )

fig.add_annotation(
    x=0.01,  # Adjust if you have multiple violins (e.g., grouped by drug)
    y=0.95,
    text=f"Median = {naive_NELLY_fixed_drug.median(1).item():.2f}",
    xref="paper", yref="paper",
    showarrow=False,
    yanchor='bottom',
    font=dict(color="black", size=12)
)

fig.add_annotation(
    x=0.2,  # Adjust if you have multiple violins (e.g., grouped by drug)
    y=0.95,
    text=f"Median = {screendl_fixed_drug.median(1).item():.2f}",
    xref="paper", yref="paper",
    showarrow=False,
    yanchor='bottom',
    font=dict(color="black", size=12)
)


fig.add_annotation(
    x=0.48,  # Adjust if you have multiple violins (e.g., grouped by drug)
    y=0.95,
    text=f"Median = {hidra_fixed_drug.median(1).item():.2f}",
    xref="paper", yref="paper",
    showarrow=False,
    yanchor='bottom',
    font=dict(color="black", size=12)
)

fig.add_annotation(
    x=0.75,  # Adjust if you have multiple violins (e.g., grouped by drug)
    y=0.95,
    text=f"Median = {paccmann_fixed_drug.median(1).item():.2f}",
    xref="paper", yref="paper",
    showarrow=False,
    yanchor='bottom',
    font=dict(color="black", size=12)
)


fig.add_annotation(
    x=0.95,  # Adjust if you have multiple violins (e.g., grouped by drug)
    y=0.95,
    text=f"Median = {dwm_NELLY_fixed_drug.median(1).item():.2f}",
    xref="paper", yref="paper",
    showarrow=False,
    yanchor='bottom',
    font=dict(color="black", size=12)
)

fig.update_yaxes(range=[-0.5, 1])

fig

In [ ]:
# fig.write_image("slides/figures_regression/benchmark/organoid_CV_fixed-drug.pdf")

## NBS cell | Fixed-line

### PCC

In [3]:
screendl_fixed_line = pd.read_csv(
    "benchmarking/screendl/predictions_pancancer_NBS_cells/pdo_fixed-line_evaluation.csv",
    header=0,
    index_col=0
)

In [4]:
screendl_fixed_line.head()

,fold_0,fold_1,fold_2,fold_3,fold_4,fold_5,fold_6,fold_7,fold_8,fold_9
ScreenDL,0.0786,0.249275,0.113607,0.027864,0.103968,0.055806,0.109403,0.022201,0.02658,0.185182


In [5]:
hidra_fixed_line = pd.read_csv(
    "benchmarking/hidra/CV/predictions_pancancer_NBS_cells/pdo_fixed-line_CV.csv",
    index_col=0
)

In [6]:
hidra_fixed_line.head()

,fold_0,fold_1,fold_2,fold_3,fold_4,fold_5,fold_6,fold_7,fold_8,fold_9
HiDRA,0.417023,0.640518,0.668338,0.63661,0.717667,0.669208,0.593802,0.692047,0.702904,0.581446


In [7]:
paccmann_fixed_line = pd.read_csv(
    "benchmarking/paccmann_predictor/CV/predictions_pancancer_NBS_cells/pdo_fixed-line_CV.csv",
    index_col=0
)

In [8]:
paccmann_fixed_line.head()

,fold_0,fold_1,fold_2,fold_3,fold_4,fold_5,fold_6,fold_7,fold_8,fold_9
Paccmann,0.45298,0.572077,0.430812,0.326795,0.282844,0.347954,0.310039,0.355408,0.213075,0.347155


In [9]:
naive_NELLY_fixed_line = pd.read_csv(
    "naive_predictor/L1000_pancancer_NBS_cells/pdo_fixed-line_CV.csv",
    index_col=0
)

In [10]:
naive_NELLY_fixed_line.columns = screendl_fixed_line.columns

In [11]:
naive_NELLY_fixed_line.head()

,fold_0,fold_1,fold_2,fold_3,fold_4,fold_5,fold_6,fold_7,fold_8,fold_9
naive-NELLY,0.716176,0.726679,0.744722,0.725847,0.734499,0.717082,0.716742,0.709232,0.718061,0.734619


In [12]:
dwm_NELLY_fixed_line = pd.read_csv(
    "cDWM/L1000_pancancer_NBS_cells/pdo_fixed-line_CV.csv",
    index_col=0
)

In [13]:
dwm_NELLY_fixed_line.columns = screendl_fixed_line.columns

In [14]:
dwm_NELLY_fixed_line.head()

,fold_0,fold_1,fold_2,fold_3,fold_4,fold_5,fold_6,fold_7,fold_8,fold_9
DWM-NELLY,0.721992,0.719065,0.703666,0.711433,0.745638,0.709545,0.740045,0.739978,0.746362,0.739592


In [15]:
metrics_fixed_line = pd.concat(
    [
        screendl_fixed_line,
        paccmann_fixed_line,
        hidra_fixed_line,
        naive_NELLY_fixed_line,
        dwm_NELLY_fixed_line
    ]
).T

In [16]:
metrics_fixed_line.reset_index(inplace=True)

In [17]:
metrics_fixed_line

,index,ScreenDL,Paccmann,HiDRA,naive-NELLY,DWM-NELLY
0,fold_0,0.078600,0.452980,0.417023,0.716176,0.721992
1,fold_1,0.249275,0.572077,0.640518,0.726679,0.719065
2,fold_2,0.113607,0.430812,0.668338,0.744722,0.703666
3,fold_3,0.027864,0.326795,0.636610,0.725847,0.711433
4,fold_4,0.103968,0.282844,0.717667,0.734499,0.745638
5,fold_5,0.055806,0.347954,0.669208,0.717082,0.709545
6,fold_6,0.109403,0.310039,0.593802,0.716742,0.740045
7,fold_7,0.022201,0.355408,0.692047,0.709232,0.739978
8,fold_8,0.026580,0.213075,0.702904,0.718061,0.746362
9,fold_9,0.185182,0.347155,0.581446,0.734619,0.739592


In [18]:
metrics_fixed_line = metrics_fixed_line.melt(
    id_vars="index",
    var_name="Method",
    value_name="MCorrelation"
)

In [19]:
metrics_fixed_line.head()

,index,Method,MCorrelation
0,fold_0,ScreenDL,0.078600
1,fold_1,ScreenDL,0.249275
2,fold_2,ScreenDL,0.113607
3,fold_3,ScreenDL,0.027864
4,fold_4,ScreenDL,0.103968


In [20]:
palette = {
    "Paccmann": "#1F77B4",       # blue
    "HiDRA": "#FF7F0E",  # orange
    "ScreenDL": "#2CA02C",     # green
    "naive-NELLY": "#D62728",   # red
    "DWM-NELLY" : "#b38ce3"
}

# _, p = wilcoxon(per_drug_pcc_cv.T["CPV"], screendl_fixed_drug.T["ScreenDL"], alternative="greater")
fig = px.violin(metrics_fixed_line,
                x="Method",
                y="MCorrelation",
                color="Method",
                height=500,
                width=700,
                template="simple_white",
                box=True,
                points="all",
                labels={"Method": "",
                        "MCorrelation": "Pearson Correlation"
                       },
                color_discrete_map=palette,
                title= "CV on organoid dataset | Fixed-line Correlation"
                
                
         )





fig.add_annotation(
    x=0,  # Adjust if you have multiple violins (e.g., grouped by drug)
    y=1,
    text=f"M = {screendl_fixed_line.median(1).item():.2f}",
    xref="paper", yref="paper",
    showarrow=False,
    yanchor='bottom',
    font=dict(color="black", size=12)
)

fig.add_annotation(
    x=0.2,  # Adjust if you have multiple violins (e.g., grouped by drug)
    y=1,
    text=f"M = {paccmann_fixed_line.median(1).item():.2f}",
    xref="paper", yref="paper",
    showarrow=False,
    yanchor='bottom',
    font=dict(color="black", size=12)
)

fig.add_annotation(
    x=0.5,  # Adjust if you have multiple violins (e.g., grouped by drug)
    y=1,
    text=f"M = {hidra_fixed_line.median(1).item():.2f}",
    xref="paper", yref="paper",
    showarrow=False,
    yanchor='bottom',
    font=dict(color="black", size=12)
)

fig.add_annotation(
    x=0.78,  # Adjust if you have multiple violins (e.g., grouped by drug)
    y=1,
    text=f"M = {naive_NELLY_fixed_line.median(1).item():.2f}",
    xref="paper", yref="paper",
    showarrow=False,
    yanchor='bottom',
    font=dict(color="black", size=12)
)


fig.add_annotation(
    x=0.98,  # Adjust if you have multiple violins (e.g., grouped by drug)
    y=1,
    text=f"M = {dwm_NELLY_fixed_line.median(1).item():.2f}",
    xref="paper", yref="paper",
    showarrow=False,
    yanchor='bottom',
    font=dict(color="black", size=12)
)

fig.update_traces(
    marker=dict(
        size=3,
        line=dict(width=2, color="DarkSlateGrey")
    )
)

fig.update_traces(
    jitter=0.05,
    pointpos=0,
    width=0.5
)


fig.update_layout(font=dict(size=16), showlegend=False)
fig.update_yaxes(range=[-0.5, 1])
fig.update_xaxes(tickangle=-90)

fig

In [21]:
fig.write_image("slides/figures_regression/benchmark/organoid_CV_fixed-line.pdf")

In [54]:
paccmann_fixed_line.median(1)

Paccmann    0.461609
dtype: float64

In [17]:
_, p = wilcoxon(
  hidra_fixed_line.T["HiDRA"],
  dwm_NELLY_fixed_line.T["DWM-NELLY"],
  alternative="two-sided"
)
p

np.float64(0.001953125)

### Precision@Q25

In [18]:
screendl_fixed_line = pd.read_csv(
    "benchmarking/screendl/predictions_pancancer_NBS_cells/precision_pdo_fixed-line_CV.csv",
    header=0,
    index_col=0
)

In [19]:
screendl_fixed_line.head()

,fold_0,fold_1,fold_2,fold_3,fold_4,fold_5,fold_6,fold_7,fold_8,fold_9
ScreenDL,0.411111,0.411111,0.411111,0.411111,0.411111,0.411111,0.411111,0.411111,0.411111,0.411111


In [20]:
hidra_fixed_line = pd.read_csv(
    "benchmarking/hidra/CV/predictions_pancancer_NBS_cells/precision_pdo_fixed-line.csv",
    index_col=0
)

In [21]:
hidra_fixed_line.head()

,fold_0,fold_1,fold_2,fold_3,fold_4,fold_5,fold_6,fold_7,fold_8,fold_9
HiDRA,0.444444,0.555556,0.555556,0.555556,0.611111,0.611111,0.555556,0.555556,0.611111,0.444444


In [22]:
paccmann_fixed_line = pd.read_csv(
    "benchmarking/paccmann_predictor/CV/predictions_pancancer_NBS_cells/precision_pdo_fixed-line_CV.csv",
    index_col=0
)

In [23]:
paccmann_fixed_line.head()

,fold_0,fold_1,fold_2,fold_3,fold_4,fold_5,fold_6,fold_7,fold_8,fold_9
Paccmann,0.555556,0.277778,0.5,0.388889,0.277778,0.375,0.375,0.444444,0.555556,0.444444


In [24]:
naive_NELLY_fixed_line = pd.read_csv(
    "naive_predictor/L1000_pancancer_NBS_cells/precision_pdo_fixed-line_CV.csv",
    index_col=0
)

In [25]:
naive_NELLY_fixed_line.columns = screendl_fixed_line.columns

In [26]:
naive_NELLY_fixed_line.head()

,fold_0,fold_1,fold_2,fold_3,fold_4,fold_5,fold_6,fold_7,fold_8,fold_9
naive-NELLY,0.611111,0.611111,0.555556,0.611111,0.611111,0.611111,0.611111,0.611111,0.666667,0.555556


In [27]:
dwm_NELLY_fixed_line = pd.read_csv(
    "cDWM/L1000_pancancer_NBS_cells/precision_pdo_fixed-line_CV.csv",
    index_col=0
)

In [28]:
dwm_NELLY_fixed_line.columns = screendl_fixed_line.columns

In [29]:
dwm_NELLY_fixed_line.head()

,fold_0,fold_1,fold_2,fold_3,fold_4,fold_5,fold_6,fold_7,fold_8,fold_9
DWM-NELLY,0.611111,0.611111,0.611111,0.611111,0.666667,0.611111,0.611111,0.611111,0.666667,0.611111


In [34]:
metrics_fixed_line = pd.concat(
    [
        screendl_fixed_line,
        paccmann_fixed_line,
        hidra_fixed_line,
        naive_NELLY_fixed_line,
        dwm_NELLY_fixed_line
    ]
).T

In [35]:
metrics_fixed_line.reset_index(inplace=True)

In [36]:
metrics_fixed_line

,index,ScreenDL,Paccmann,HiDRA,naive-NELLY,DWM-NELLY
0,fold_0,0.411111,0.555556,0.444444,0.611111,0.611111
1,fold_1,0.411111,0.277778,0.555556,0.611111,0.611111
2,fold_2,0.411111,0.500000,0.555556,0.555556,0.611111
3,fold_3,0.411111,0.388889,0.555556,0.611111,0.611111
4,fold_4,0.411111,0.277778,0.611111,0.611111,0.666667
5,fold_5,0.411111,0.375000,0.611111,0.611111,0.611111
6,fold_6,0.411111,0.375000,0.555556,0.611111,0.611111
7,fold_7,0.411111,0.444444,0.555556,0.611111,0.611111
8,fold_8,0.411111,0.555556,0.611111,0.666667,0.666667
9,fold_9,0.411111,0.444444,0.444444,0.555556,0.611111


In [37]:
metrics_fixed_line = metrics_fixed_line.melt(
    id_vars="index",
    var_name="Method",
    value_name="MPrecision"
)

In [38]:
metrics_fixed_line.head()

,index,Method,MPrecision
0,fold_0,ScreenDL,0.411111
1,fold_1,ScreenDL,0.411111
2,fold_2,ScreenDL,0.411111
3,fold_3,ScreenDL,0.411111
4,fold_4,ScreenDL,0.411111


In [39]:
palette = {
    "Paccmann": "#1F77B4",       # blue
    "HiDRA": "#FF7F0E",  # orange
    "ScreenDL": "#2CA02C",     # green
    "naive-NELLY": "#D62728",   # red
    "DWM-NELLY" : "#b38ce3"
}

# _, p = wilcoxon(per_drug_pcc_cv.T["CPV"], screendl_fixed_drug.T["ScreenDL"], alternative="greater")
fig = px.violin(metrics_fixed_line,
                x="Method",
                y="MPrecision",
                color="Method",
                height=500,
                width=700,
                template="simple_white",
                box=True,
                points="all",
                labels={"Method": "",
                        "MPrecision": "Precision@Q25"
                       },
                color_discrete_map=palette,
                title= "CV on organoid dataset | Fixed-line precision@Q25"
                
                
         )





fig.add_annotation(
    x=0,  # Adjust if you have multiple violins (e.g., grouped by drug)
    y=1,
    text=f"M = {screendl_fixed_line.median(1).item():.2f}",
    xref="paper", yref="paper",
    showarrow=False,
    yanchor='bottom',
    font=dict(color="black", size=12)
)

fig.add_annotation(
    x=0.2,  # Adjust if you have multiple violins (e.g., grouped by drug)
    y=1,
    text=f"M = {paccmann_fixed_line.median(1).item():.2f}",
    xref="paper", yref="paper",
    showarrow=False,
    yanchor='bottom',
    font=dict(color="black", size=12)
)

fig.add_annotation(
    x=0.5,  # Adjust if you have multiple violins (e.g., grouped by drug)
    y=1,
    text=f"M = {hidra_fixed_line.median(1).item():.2f}",
    xref="paper", yref="paper",
    showarrow=False,
    yanchor='bottom',
    font=dict(color="black", size=12)
)

fig.add_annotation(
    x=0.78,  # Adjust if you have multiple violins (e.g., grouped by drug)
    y=1,
    text=f"M = {naive_NELLY_fixed_line.median(1).item():.2f}",
    xref="paper", yref="paper",
    showarrow=False,
    yanchor='bottom',
    font=dict(color="black", size=12)
)


fig.add_annotation(
    x=0.98,  # Adjust if you have multiple violins (e.g., grouped by drug)
    y=1,
    text=f"M = {dwm_NELLY_fixed_line.median(1).item():.2f}",
    xref="paper", yref="paper",
    showarrow=False,
    yanchor='bottom',
    font=dict(color="black", size=12)
)

fig.update_traces(
    marker=dict(
        size=3,
        line=dict(width=2, color="DarkSlateGrey")
    )
)

fig.update_traces(
    jitter=0.05,
    pointpos=0,
    width=0.5
)


fig.update_layout(font=dict(size=16), showlegend=False)
fig.update_yaxes(range=[-0.5, 1])
fig.update_xaxes(tickangle=-90)

fig

In [40]:
# fig.write_image("slides/figures_regression/benchmark/organoid_precision_CV_fixed-line.pdf")

In [32]:
_, p = wilcoxon(
  hidra_fixed_line.T["HiDRA"],
  dwm_NELLY_fixed_line.T["DWM-NELLY"],
  alternative="two-sided"
)
p

np.float64(0.00390625)

### NDCG@Q25

In [33]:
screendl_fixed_line = pd.read_csv(
    "benchmarking/screendl/predictions_pancancer_NBS_cells/ndcg_pdo_fixed-line_CV.csv",
    header=0,
    index_col=0
)

In [34]:
screendl_fixed_line.head()

,fold_0,fold_1,fold_2,fold_3,fold_4,fold_5,fold_6,fold_7,fold_8,fold_9
ScreenDL,0.321888,0.321888,0.321888,0.321888,0.321888,0.321888,0.321888,0.321888,0.321888,0.321888


In [35]:
hidra_fixed_line = pd.read_csv(
    "benchmarking/hidra/CV/predictions_pancancer_NBS_cells/ndcg_pdo_fixed-line.csv",
    index_col=0
)

In [36]:
hidra_fixed_line.head()

,fold_0,fold_1,fold_2,fold_3,fold_4,fold_5,fold_6,fold_7,fold_8,fold_9
HiDRA,0.0,0.220812,0.237623,0.20754,0.152446,0.253367,0.305846,0.253533,0.268474,0.106988


In [37]:
paccmann_fixed_line = pd.read_csv(
    "benchmarking/paccmann_predictor/CV/predictions_pancancer_NBS_cells/ndcg_pdo_fixed-line_CV.csv",
    index_col=0
)

In [38]:
paccmann_fixed_line.head()

,fold_0,fold_1,fold_2,fold_3,fold_4,fold_5,fold_6,fold_7,fold_8,fold_9
Paccmann,0.662268,0.169084,0.477422,0.424746,0.089658,0.462213,0.491289,0.380373,0.622471,0.40892


In [39]:
naive_NELLY_fixed_line = pd.read_csv(
    "naive_predictor/L1000_pancancer_NBS_cells/ndcg_pdo_fixed-line_CV.csv",
    index_col=0
)

In [40]:
naive_NELLY_fixed_line.columns = screendl_fixed_line.columns

In [41]:
naive_NELLY_fixed_line.head()

,fold_0,fold_1,fold_2,fold_3,fold_4,fold_5,fold_6,fold_7,fold_8,fold_9
naive-NELLY,0.751529,0.762206,0.7338,0.751981,0.755445,0.74931,0.76781,0.751768,0.797533,0.746167


In [42]:
dwm_NELLY_fixed_line = pd.read_csv(
    "cDWM/L1000_pancancer_NBS_cells/ndcg_pdo_fixed-line_CV.csv",
    index_col=0
)

In [43]:
dwm_NELLY_fixed_line.columns = screendl_fixed_line.columns

In [44]:
dwm_NELLY_fixed_line.head()

,fold_0,fold_1,fold_2,fold_3,fold_4,fold_5,fold_6,fold_7,fold_8,fold_9
DWM-NELLY,0.751394,0.724845,0.785638,0.729603,0.797533,0.772928,0.761222,0.765154,0.797533,0.761222


In [65]:
metrics_fixed_line = pd.concat(
    [
        hidra_fixed_line,
        screendl_fixed_line,
        paccmann_fixed_line,
        naive_NELLY_fixed_line,
        dwm_NELLY_fixed_line
    ]
).T

In [66]:
metrics_fixed_line.reset_index(inplace=True)

In [67]:
metrics_fixed_line

,index,HiDRA,ScreenDL,Paccmann,naive-NELLY,DWM-NELLY
0,fold_0,0.000000,0.321888,0.662268,0.751529,0.751394
1,fold_1,0.220812,0.321888,0.169084,0.762206,0.724845
2,fold_2,0.237623,0.321888,0.477422,0.733800,0.785638
3,fold_3,0.207540,0.321888,0.424746,0.751981,0.729603
4,fold_4,0.152446,0.321888,0.089658,0.755445,0.797533
5,fold_5,0.253367,0.321888,0.462213,0.749310,0.772928
6,fold_6,0.305846,0.321888,0.491289,0.767810,0.761222
7,fold_7,0.253533,0.321888,0.380373,0.751768,0.765154
8,fold_8,0.268474,0.321888,0.622471,0.797533,0.797533
9,fold_9,0.106988,0.321888,0.408920,0.746167,0.761222


In [68]:
metrics_fixed_line = metrics_fixed_line.melt(
    id_vars="index",
    var_name="Method",
    value_name="MNDCG"
)

In [69]:
metrics_fixed_line.head()

,index,Method,MNDCG
0,fold_0,HiDRA,0.000000
1,fold_1,HiDRA,0.220812
2,fold_2,HiDRA,0.237623
3,fold_3,HiDRA,0.207540
4,fold_4,HiDRA,0.152446


In [70]:
palette = {
    "Paccmann": "#1F77B4",       # blue
    "HiDRA": "#FF7F0E",  # orange
    "ScreenDL": "#2CA02C",     # green
    "naive-NELLY": "#D62728",   # red
    "DWM-NELLY" : "#b38ce3"
}

# _, p = wilcoxon(per_drug_pcc_cv.T["CPV"], screendl_fixed_drug.T["ScreenDL"], alternative="greater")
fig = px.violin(metrics_fixed_line,
                x="Method",
                y="MNDCG",
                color="Method",
                height=500,
                width=700,
                template="simple_white",
                box=True,
                points="all",
                labels={"Method": "",
                        "MNDCG": "NDCG@Q25"
                       },
                color_discrete_map=palette,
                title= "CV on organoid dataset | Fixed-line NDCG@Q25"
                
                
         )




fig.add_annotation(
    x=0,  # Adjust if you have multiple violins (e.g., grouped by drug)
    y=1,
    text=f"M = {hidra_fixed_line.median(1).item():.2f}",
    xref="paper", yref="paper",
    showarrow=False,
    yanchor='bottom',
    font=dict(color="black", size=12)
)

fig.add_annotation(
    x=0.2,  # Adjust if you have multiple violins (e.g., grouped by drug)
    y=1,
    text=f"M = {screendl_fixed_line.median(1).item():.2f}",
    xref="paper", yref="paper",
    showarrow=False,
    yanchor='bottom',
    font=dict(color="black", size=12)
)


fig.add_annotation(
    x=0.5,  # Adjust if you have multiple violins (e.g., grouped by drug)
    y=1,
    text=f"M = {paccmann_fixed_line.median(1).item():.2f}",
    xref="paper", yref="paper",
    showarrow=False,
    yanchor='bottom',
    font=dict(color="black", size=12)
)


fig.add_annotation(
    x=0.78,  # Adjust if you have multiple violins (e.g., grouped by drug)
    y=1,
    text=f"M = {naive_NELLY_fixed_line.median(1).item():.2f}",
    xref="paper", yref="paper",
    showarrow=False,
    yanchor='bottom',
    font=dict(color="black", size=12)
)


fig.add_annotation(
    x=0.98,  # Adjust if you have multiple violins (e.g., grouped by drug)
    y=1,
    text=f"M = {dwm_NELLY_fixed_line.median(1).item():.2f}",
    xref="paper", yref="paper",
    showarrow=False,
    yanchor='bottom',
    font=dict(color="black", size=12)
)

fig.update_traces(
    marker=dict(
        size=3,
        line=dict(width=2, color="DarkSlateGrey")
    )
)

fig.update_traces(
    jitter=0.05,
    pointpos=0,
    width=0.5
)


fig.update_layout(font=dict(size=16), showlegend=False)
fig.update_yaxes(range=[-0.5, 1])
fig.update_xaxes(tickangle=-90)

fig

In [71]:
# fig.write_image("slides/figures_regression/benchmark/organoid_ndcg_CV_fixed-line.pdf")

In [45]:
_, p = wilcoxon(
  paccmann_fixed_line.T["Paccmann"],
  dwm_NELLY_fixed_line.T["DWM-NELLY"],
  alternative="two-sided"
)
p

np.float64(0.001953125)

## NBS cell | Fixed-cancer type

In [53]:
screendl_fixed_cancer = pd.read_csv(
    "benchmarking/screendl/predictions_pancancer_NBS_cells/pdo_fixed-cancer_CV.csv",
    index_col=0
).T

screendl_fixed_cancer

,BLCA,COAD,HNSC,PDAC
ScreenDL,0.209691,0.020824,0.194371,0.095119


In [54]:
hidra_fixed_cancer = pd.read_csv(
    "benchmarking/hidra/CV/predictions_pancancer_NBS_cells/pdo_fixed-cancer_CV.csv",
    index_col=0
).T

hidra_fixed_cancer

,BLCA,COAD,HNSC,PDAC
HiDRA,0.166064,0.618579,0.285354,0.737829


In [55]:
paccmann_fixed_cancer = pd.read_csv(
    "benchmarking/paccmann_predictor/CV/predictions_pancancer_NBS_cells/pdo_fixed-cancer_CV.csv",
    index_col=0
).T

paccmann_fixed_cancer

,BLCA,COAD,HNSC,PDAC
Paccmann,0.202282,0.320584,0.143083,0.599227


In [56]:
naive_NELLY_fixed_cancer = pd.read_csv(
    "naive_predictor/L1000_pancancer_NBS_cells/pdo_fixed-cancer_CV.csv",
    index_col=0,
).T

naive_NELLY_fixed_cancer

,BLCA,COAD,HNSC,PDAC
naive-NELLY,0.26781,0.755169,0.321104,0.838369


In [57]:
dwm_NELLY_fixed_cancer = pd.read_csv(
    "cDWM/L1000_pancancer_NBS_cells/pdo_fixed-cancer_CV.csv",
    index_col=0
).T

dwm_NELLY_fixed_cancer

,BLCA,COAD,HNSC,PDAC
DWM-NELLY,0.268545,0.75499,0.330354,0.834913


In [58]:
metrics_fixed_cancer = pd.concat(
    [
        screendl_fixed_cancer,
        hidra_fixed_cancer,
        paccmann_fixed_cancer,
        naive_NELLY_fixed_cancer,
        dwm_NELLY_fixed_cancer
    ], axis=0
)

metrics_fixed_cancer = metrics_fixed_cancer.reset_index()

In [59]:
metrics_fixed_cancer = metrics_fixed_cancer.melt(
    id_vars="index",
    var_name="cancer_type",
    value_name="MCorrelation"
)

metrics_fixed_cancer.head()

,index,cancer_type,MCorrelation
0,ScreenDL,BLCA,0.209691
1,HiDRA,BLCA,0.166064
2,Paccmann,BLCA,0.202282
3,naive-NELLY,BLCA,0.267810
4,DWM-NELLY,BLCA,0.268545


In [60]:
# _, p = wilcoxon(per_drug_pcc_cv.T["CPV"], screendl_fixed_drug.T["ScreenDL"], alternative="greater")

palette = {
    "Paccmann": "#1F77B4",       # blue
    "HiDRA": "#FF7F0E",  # orange
    "ScreenDL": "#2CA02C",     # green
    "naive-NELLY": "#D62728",   # red
    "DWM-NELLY" : "#b38ce3"
}

symbol_map = {
    "Paccmann": "square",        # ■
    "HiDRA": "diamond",          # ◆
    "ScreenDL": "x",      # ✱-like
    "naive-NELLY": "cross",      # +
    "DWM-NELLY": "circle",  # ○
}

fig = px.scatter(metrics_fixed_cancer,
                x="cancer_type",
                y="MCorrelation",
                color="index",
                height=600,
                width=700,
                symbol="index",
                 symbol_map=symbol_map,
                template="simple_white",
                category_orders={"index": ["Paccmann", "HiDRA", "ScreenDL", "naive-NELLY", "DWM-NELLY"]},
                labels={"cancer_type": "",
                        "MCorrelation": "Pearson Correlation",
                        "index" : ""
                       },
                color_discrete_map=palette,
                title= "CV on NBS cells | Fixed-drug grouped by pathway"
                
                
         )


# fig.update_traces(jitter=0.05, pointpos=0)

fig.update_traces(
    marker=dict(
        size=10,
        line=dict(width=2, color="DarkSlateGrey")
    )
)
fig.update_layout(font=dict(size=16))
fig.update_yaxes(range=[-0.5, 1])
fig.update_xaxes(tickangle=45) 

fig.update_layout(
    legend=dict(
        orientation="h",
        yanchor="top",
        y=1.1,
        xanchor="center",
        x=0.5
    )
)

fig

In [61]:
fig.write_image(
    "slides/figures_regression/benchmark/organoid_CV_fixed-cancer_type.pdf"
)

# PDO benchmark descriptive figures

In [12]:
pdo_plot.reset_index()

,TCGA_DESC,count
0,HNSC,19
1,COAD,19
2,BLCA,11
3,PDAC,5


In [100]:
pdo_plot = (
    pdo_dr
    .drop_duplicates(subset="Line")
    .TCGA_DESC
    .value_counts()
    .reset_index()
)


fig = px.pie(
    pdo_plot,
    values="count",
    names="TCGA_DESC",
    hole=.7,
    color_discrete_sequence=px.colors.qualitative.Dark2,
    width=600,
    height=600
    )

fig.update_traces(
    texttemplate="%{value}",   # show raw counts
    textposition="inside"
)

fig.update_layout(
    font=dict(size=16)
)

fig

In [101]:
fig.write_image(
    "slides/figures_regression/benchmark/benchmark_dataset_PDOS.pdf"
)

In [51]:
pdo_dr.head()

,Line,Drug,LogIC50,PATHWAY_NAME,seen_before,Lab,TCGA_DESC,Y_TRUE
0,P9-M-LN,Cisplatin,2.165619,DNA replication,yes,KK,HNSC,-0.404912
1,P20-T-G,Cisplatin,1.249902,DNA replication,yes,KK,HNSC,-1.353180
2,P47-T-T,Cisplatin,1.047319,DNA replication,yes,KK,HNSC,-1.000000
3,P47-R-AP,Cisplatin,1.420696,DNA replication,yes,KK,HNSC,-0.110678
4,P48-M-LN,Cisplatin,1.667707,DNA replication,yes,KK,HNSC,-1.602422


In [68]:
(
    pdo_dr
    .groupby(["TCGA_DESC", "PATHWAY_NAME"])["Drug"]
    .nunique()
    .reset_index()
)

,TCGA_DESC,PATHWAY_NAME,Drug
0,BLCA,Apoptosis regulation,2
1,BLCA,Cell cycle,2
2,BLCA,Chromatin other,1
3,BLCA,DNA replication,4
4,BLCA,EGFR signaling,2
...,...,...,...
59,PDAC,PI3K/MTOR signaling,10
60,PDAC,Protein stability and degradation,2
61,PDAC,RTK signaling,6
62,PDAC,WNT signaling,2


In [7]:
# pdo_plot = (
#     pdo_dr[["TCGA_DESC", "PATHWAY_NAME"]]
#     .value_counts()
#     .reset_index()
# )


pdo_plot = (
    pdo_dr
    .groupby(["TCGA_DESC", "PATHWAY_NAME"])["Drug"]
    .nunique()
    .reset_index()
)

fig = px.scatter(
    pdo_plot,
    x="TCGA_DESC",
    y="PATHWAY_NAME",
    template="simple_white",
    size="Drug",
    color="Drug",
    color_continuous_scale=px.colors.sequential.Greens,
    width=700,
    height=900,
    labels={
        "PATHWAY_NAME": "Target pathway",
        "TCGA_DESC": "Cancer type",
        "Drug": "Nº drugs"
    }
)

fig.update_traces(
    marker=dict(
        line=dict(width=1, color="DarkSlateGrey"),
    )
)

fig.update_layout(
    font=dict(size=16)
)

fig

In [8]:
# fig.write_image(
#     "slides/figures_regression/benchmark/benchmark_dataset_drug_coverage.pdf"
# )